In [2]:
# ✅ Install library jika diperlukan
# !pip install tensorflow numpy matplotlib

import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)

##############################################
# 1️⃣ Simple Linear Autoencoder (PCA Example)
##############################################
print("\n=== 1. Simple Linear Autoencoder ===")

# Membuat encoder dan decoder
encoder = keras.models.Sequential([
    keras.layers.Dense(2, input_shape=[3], name="encoder_dense")
])

decoder = keras.models.Sequential([
    keras.layers.Dense(3, input_shape=[2], name="decoder_dense")
])

# Gabungkan encoder dan decoder
autoencoder = keras.models.Sequential([encoder, decoder])
autoencoder.compile(loss="mse", optimizer=keras.optimizers.SGD(learning_rate=0.1))

# Generate dummy 3D dataset
np.random.seed(42)
X_train = np.random.rand(1000, 3)

print("Training simple autoencoder...")
history_simple = autoencoder.fit(X_train, X_train, epochs=20, verbose=0)
codings = encoder.predict(X_train, verbose=0)
print(f"Original data shape: {X_train.shape}")
print(f"Encoded data shape: {codings.shape}")
print(f"Final loss: {history_simple.history['loss'][-1]:.4f}")

##############################################
# 2️⃣ Stacked Autoencoder (Fashion MNIST Example)
##############################################
print("\n=== 2. Stacked Autoencoder (Fashion MNIST) ===")

# Load dan preprocess data
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train_full = X_train_full.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_valid.shape}")

# Membuat stacked autoencoder
stacked_encoder = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(100, activation="selu"),
    keras.layers.Dense(30, activation="selu"),
], name="stacked_encoder")

stacked_decoder = keras.models.Sequential([
    keras.layers.Dense(100, activation="selu", input_shape=[30]),
    keras.layers.Dense(28 * 28, activation="sigmoid"),
    keras.layers.Reshape([28, 28])
], name="stacked_decoder")

stacked_ae = keras.models.Sequential([stacked_encoder, stacked_decoder])
stacked_ae.compile(loss="binary_crossentropy", optimizer=keras.optimizers.SGD(learning_rate=1.5))

print("Training stacked autoencoder...")
history_stacked = stacked_ae.fit(X_train, X_train, epochs=10,
                                validation_data=(X_valid, X_valid), verbose=1)

##############################################
# 3️⃣ Convolutional Autoencoder
##############################################
print("\n=== 3. Convolutional Autoencoder ===")

# Membuat convolutional encoder
conv_encoder = keras.models.Sequential([
    keras.layers.Reshape([28, 28, 1], input_shape=[28, 28]),
    keras.layers.Conv2D(16, 3, padding="same", activation="selu"),
    keras.layers.MaxPool2D(2),
    keras.layers.Conv2D(32, 3, padding="same", activation="selu"),
    keras.layers.MaxPool2D(2),
    keras.layers.Conv2D(64, 3, padding="same", activation="selu"),
    keras.layers.MaxPool2D(2),
], name="conv_encoder")

# Membuat convolutional decoder
conv_decoder = keras.models.Sequential([
    keras.layers.Conv2DTranspose(32, 3, strides=2, padding="valid", activation="selu",
                                input_shape=[3, 3, 64]),
    keras.layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="selu"),
    keras.layers.Conv2DTranspose(1, 3, strides=2, padding="same", activation="sigmoid"),
    keras.layers.Reshape([28, 28])
], name="conv_decoder")

conv_ae = keras.models.Sequential([conv_encoder, conv_decoder])
conv_ae.compile(loss="binary_crossentropy", optimizer="adam")

print("Training convolutional autoencoder...")
history_conv = conv_ae.fit(X_train, X_train, epochs=5,
                          validation_data=(X_valid, X_valid), verbose=1)

##############################################
# 4️⃣ Variational Autoencoder (VAE)
##############################################
print("\n=== 4. Variational Autoencoder (VAE) ===")

class Sampling(keras.layers.Layer):
    def call(self, inputs):
        mean, log_var = inputs
        batch_size = tf.shape(mean)[0]
        dim = tf.shape(mean)[1]
        epsilon = tf.random.normal(shape=(batch_size, dim))
        return mean + tf.exp(0.5 * log_var) * epsilon

# VAE parameters
codings_size = 10

# Encoder
inputs = keras.layers.Input(shape=[28, 28])
z = keras.layers.Flatten()(inputs)
z = keras.layers.Dense(150, activation="selu")(z)
z = keras.layers.Dense(100, activation="selu")(z)
codings_mean = keras.layers.Dense(codings_size)(z)
codings_log_var = keras.layers.Dense(codings_size)(z)
codings = Sampling()([codings_mean, codings_log_var])

variational_encoder = keras.Model(inputs=[inputs], outputs=[codings_mean, codings_log_var, codings])

# Decoder
decoder_inputs = keras.layers.Input(shape=[codings_size])
x = keras.layers.Dense(100, activation="selu")(decoder_inputs)
x = keras.layers.Dense(150, activation="selu")(x)
x = keras.layers.Dense(28 * 28, activation="sigmoid")(x)
outputs = keras.layers.Reshape([28, 28])(x)
variational_decoder = keras.Model(inputs=[decoder_inputs], outputs=[outputs])

# VAE Model
_, _, codings = variational_encoder(inputs)
reconstructions = variational_decoder(codings)
variational_ae = keras.Model(inputs=[inputs], outputs=[reconstructions])

# KL divergence loss
latent_loss = -0.5 * tf.reduce_sum(
    1 + codings_log_var - tf.square(codings_mean) - tf.exp(codings_log_var), axis=-1)
variational_ae.add_loss(tf.reduce_mean(latent_loss) / 784.0)

variational_ae.compile(loss="binary_crossentropy", optimizer="rmsprop", metrics=["accuracy"])

print("Training variational autoencoder...")
history_vae = variational_ae.fit(X_train, X_train, epochs=15, batch_size=128,
                                validation_data=(X_valid, X_valid), verbose=1)

##############################################
# 5️⃣ Generative Adversarial Networks (GAN)
##############################################
print("\n=== 5. Generative Adversarial Networks (GAN) ===")

codings_size = 30

# Generator
generator = keras.models.Sequential([
    keras.layers.Dense(100, activation="selu", input_shape=[codings_size]),
    keras.layers.Dense(150, activation="selu"),
    keras.layers.Dense(28 * 28, activation="sigmoid"),
    keras.layers.Reshape([28, 28])
], name="generator")

# Discriminator
discriminator = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(150, activation="selu"),
    keras.layers.Dense(100, activation="selu"),
    keras.layers.Dense(1, activation="sigmoid")
], name="discriminator")

# Compile discriminator
discriminator.compile(loss="binary_crossentropy", optimizer="rmsprop")

# Create GAN
discriminator.trainable = False
gan = keras.models.Sequential([generator, discriminator])
gan.compile(loss="binary_crossentropy", optimizer="rmsprop")

# Training function
def train_gan(gan, dataset, batch_size, codings_size, n_epochs=5):
    generator, discriminator = gan.layers

    for epoch in range(n_epochs):
        print(f"Epoch {epoch + 1}/{n_epochs}")

        for batch_idx, X_batch in enumerate(dataset):
            # Train discriminator
            noise = tf.random.normal(shape=[batch_size, codings_size])
            generated_images = generator(noise)

            X_fake_and_real = tf.concat([generated_images, X_batch], axis=0)
            y1 = tf.concat([tf.zeros((batch_size, 1)), tf.ones((batch_size, 1))], axis=0)

            discriminator.trainable = True
            d_loss = discriminator.train_on_batch(X_fake_and_real, y1)

            # Train generator
            noise = tf.random.normal(shape=[batch_size, codings_size])
            y2 = tf.ones((batch_size, 1))

            discriminator.trainable = False
            g_loss = gan.train_on_batch(noise, y2)

            if batch_idx % 100 == 0:
                print(f"  Batch {batch_idx}: D_loss={d_loss:.4f}, G_loss={g_loss:.4f}")

# Prepare dataset
batch_size = 32
dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(1000).batch(batch_size, drop_remainder=True).prefetch(1)

print("Training GAN...")
train_gan(gan, dataset, batch_size, codings_size, n_epochs=3)

##############################################
# 6️⃣ Visualization and Results
##############################################
print("\n=== 6. Visualization and Results ===")

# Function to display images
def display_images(images, titles, rows=2, cols=5):
    fig, axes = plt.subplots(rows, cols, figsize=(12, 6))
    for i, (img, title) in enumerate(zip(images, titles)):
        row, col = i // cols, i % cols
        axes[row, col].imshow(img, cmap="binary")
        axes[row, col].set_title(title)
        axes[row, col].axis("off")
    plt.tight_layout()
    plt.show()

# Test images
test_images = X_valid[:10]

# Generate reconstructions from different autoencoders
stacked_reconstructions = stacked_ae.predict(test_images, verbose=0)
conv_reconstructions = conv_ae.predict(test_images, verbose=0)
vae_reconstructions = variational_ae.predict(test_images, verbose=0)

# Generate new images from GAN
noise = tf.random.normal(shape=[10, codings_size])
gan_generated = generator.predict(noise, verbose=0)

# Display results
print("Displaying original vs reconstructed images...")
display_images(
    images=[test_images[0], stacked_reconstructions[0], conv_reconstructions[0],
            vae_reconstructions[0], gan_generated[0]] * 2,
    titles=["Original", "Stacked AE", "Conv AE", "VAE", "GAN Generated"] * 2
)

# Plot training histories
plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
plt.plot(history_simple.history['loss'])
plt.title('Simple Autoencoder Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(2, 3, 2)
plt.plot(history_stacked.history['loss'], label='Training')
plt.plot(history_stacked.history['val_loss'], label='Validation')
plt.title('Stacked Autoencoder Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(2, 3, 3)
plt.plot(history_conv.history['loss'], label='Training')
plt.plot(history_conv.history['val_loss'], label='Validation')
plt.title('Convolutional Autoencoder Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(2, 3, 4)
plt.plot(history_vae.history['loss'], label='Training')
plt.plot(history_vae.history['val_loss'], label='Validation')
plt.title('VAE Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

print("\n=== Summary ===")
print("✅ All models trained successfully!")
print("- Simple Autoencoder: Dimensionality reduction from 3D to 2D")
print("- Stacked Autoencoder: Deep autoencoder for Fashion-MNIST")
print("- Convolutional Autoencoder: Uses CNN layers for better image reconstruction")
print("- Variational Autoencoder: Probabilistic encoder with latent space regularization")
print("- GAN: Generative model that learns to create new Fashion-MNIST images")

TensorFlow version: 2.18.0

=== 1. Simple Linear Autoencoder ===
Training simple autoencoder...
Original data shape: (1000, 3)
Encoded data shape: (1000, 2)
Final loss: 0.0288

=== 2. Stacked Autoencoder (Fashion MNIST) ===
Training set shape: (55000, 28, 28)
Validation set shape: (5000, 28, 28)
Training stacked autoencoder...
Epoch 1/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.3783 - val_loss: 0.3067
Epoch 2/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.3067 - val_loss: 0.2988
Epoch 3/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.2973 - val_loss: 0.2913
Epoch 4/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 0.2943 - val_loss: 0.2879
Epoch 5/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.2922 - val_loss: 0.2880
Epoch 6/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.2891 - val_loss: 0.2848
Epoch 7/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.2869 - val_loss: 0.2841
Epoch 8/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step 

ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.operations`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```


# 📚 Chapter 17 - Representation Learning and Generative Learning Using Autoencoders and GANs

---

## 🔸 1. Apa Itu Representation Learning?
Representation Learning → model belajar **representasi fitur yang lebih baik** secara otomatis dari data mentah.
- ✅ Berguna untuk mengurangi **dimensionalitas** atau **feature extraction**
- ✅ Bisa digunakan sebagai preprocessing untuk supervised learning

---

## 🔸 2. Autoencoders
Autoencoder → neural network **unsupervised** untuk merekonstruksi input.
- ✅ Struktur: **Encoder → Bottleneck → Decoder**
- ✅ Tujuan: output ≈ input
- ✅ Bottleneck → memaksa model menyimpan informasi paling penting → mirip dengan PCA untuk data non-linear

---

## 🔸 3. Variasi Autoencoder
| Jenis                  | Keterangan                                          |
| ---------------------- | -------------------------------------------------- |
| **Stacked Autoencoder** | Beberapa layer → belajar representasi non-linear   |
| **Convolutional AE**    | Untuk gambar → convolutional encoder-decoder       |
| **Sparse AE**          | Regularisasi sparsity → aktifkan neuron seminimal mungkin |
| **Denoising AE**       | Belajar merekonstruksi input dari input yang “rusak” |
| **Variational AE (VAE)**| Probabilistic AE → belajar distribusi latennya     |

---

## 🔸 4. Variational Autoencoders (VAE)
- ✅ Alih-alih hanya membuat titik kode → **VAE belajar distribusi probabilistik** di ruang laten.
- ✅ Bisa **menghasilkan data baru** → sampling dari distribusi laten.
- ✅ Cocok untuk **generative models** → pengembangan ke arah GAN.

---

## 🔸 5. Generative Adversarial Networks (GANs)
GAN → dua model neural network saling bertanding:
| Komponen       | Fungsi                          |
| -------------- | ------------------------------- |
| **Generator**  | Menghasilkan data palsu         |
| **Discriminator** | Menilai apakah data asli/palsu |

→ Generator belajar membuat data **semakin mirip** data nyata → Discriminator berusaha **mendeteksi kebohongan**.

---

## 🔸 6. Training GAN → Adversarial Process
1. Generator → menghasilkan gambar palsu
2. Discriminator → dilatih membedakan nyata/palsu
3. Generator → dilatih agar bisa menipu Discriminator

➡️ **Iterasi bertahap** → Generator semakin realistis → Discriminator semakin cermat.

---

## 🔸 7. Pengembangan Lanjutan
- **DCGAN** → Deep Convolutional GAN → untuk gambar
- **Conditional GAN** → dapat mengontrol output (misal, angka tertentu)
- **WGAN** → stabilisasi training
- **StyleGAN, BigGAN** → GAN paling mutakhir → menghasilkan gambar realistik berkualitas tinggi

---

## ✅ Kesimpulan
- ✅ **Autoencoder** → *Feature Extraction, Compression*
- ✅ **Variational Autoencoder** → *Generative Models berbasis probabilitas*
- ✅ **GAN** → *Model generatif paling kuat → menghasilkan gambar, audio, video, dll*

